# Inasistencias por seguimiento, grupo, programa y sede

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
# Programas excluidos de la analítica
EXCLUIR_PROGRAMAS = [
    "TecLab-AuxAdmin",
    "TecLab-MarkDig",
    "CIES-Univ",
    "CIES-RAF-FICAT",
    "DiploHASPS-1",
    "DiploHCNSPS-1",
]

In [ ]:
detalle = pd.read_csv("../data/raw/inasistencias_detalle.csv")
est = pd.read_csv("../data/raw/estudiantes.csv")

print(f"Inasistencias detalle: {len(detalle):,}")
print(f"Estudiantes: {len(est):,}")
detalle.head(2)

In [ ]:
# Merge con datos de estudiantes (programa, sede)
info_est = est[["Numero_identificacion", "Codigo_programa", "Nombre_programa_limpio", "Sede"]].drop_duplicates(
    subset="Numero_identificacion"
)

df = detalle.merge(
    info_est,
    left_on="Numero_identificacion_estudiante",
    right_on="Numero_identificacion",
    how="left",
)

df["Fecha_inasistencia"] = pd.to_datetime(df["Fecha_inasistencia"], errors="coerce")
print(f"Sin datos de programa/sede: {df['Nombre_programa_limpio'].isna().sum():,}")
df.head(2)

In [ ]:
# Filtrar programas excluidos
antes = len(df)
df = df[~df["Codigo_programa"].isin(EXCLUIR_PROGRAMAS)].copy()
print(f"Registros eliminados: {antes - len(df):,}")
print(f"Registros después del filtro: {len(df):,}")

In [ ]:
# Configuración de grupos y fechas
PROGRAMAS_GRUPO_B = ["TECNOLOGÍA EN GESTIÓN DE PRODUCCIÓN DE MODAS"]
SEDES_GRUPO_B = ["MINCA", "BURITACA"]

CORTES_A = pd.to_datetime(["2026-02-01", "2026-04-02", "2026-05-14", "2026-06-26"])
ETIQUETAS_A = [
    "Seguimiento 1 (01 Feb – 01 Abr)",
    "Seguimiento 2 (02 Abr – 13 May)",
    "Seguimiento 3 (14 May – 25 Jun)",
]

CORTES_B = pd.to_datetime(["2026-04-06", "2026-05-10", "2026-06-14", "2026-07-26"])
ETIQUETAS_B = [
    "Seguimiento 1 (06 Abr – 09 May)",
    "Seguimiento 2 (10 May – 13 Jun)",
    "Seguimiento 3 (14 Jun – 25 Jul)",
]

In [ ]:
# Asignar grupo (A o B) según programa o sede
def asignar_grupo(row):
    prog = str(row.get("Nombre_programa_limpio", "") or "").strip().upper()
    sede = str(row.get("Sede", "") or "").strip().upper()
    if prog in [p.upper() for p in PROGRAMAS_GRUPO_B] or sede in [s.upper() for s in SEDES_GRUPO_B]:
        return "B"
    return "A"

df["Grupo"] = df.apply(asignar_grupo, axis=1)

In [ ]:
# Clasificar cada falta en un seguimiento según su fecha y grupo
def clasificar_seguimiento(fecha, grupo):
    cortes = CORTES_B if grupo == "B" else CORTES_A
    etiquetas = ETIQUETAS_B if grupo == "B" else ETIQUETAS_A
    for i in range(len(etiquetas)):
        if cortes[i] <= fecha < cortes[i + 1]:
            return etiquetas[i]
    return None

df["Seguimiento"] = df.apply(
    lambda r: clasificar_seguimiento(r["Fecha_inasistencia"], r["Grupo"])
    if pd.notna(r["Fecha_inasistencia"])
    else None,
    axis=1,
)

print(f"Registros totales: {len(df):,}")
print(f"Sin seguimiento (fuera de rango): {df['Seguimiento'].isna().sum():,}")
df.head(2)

## 1. Inasistencias por grupo (A / B)

In [ ]:
por_grupo = df.groupby("Grupo").size().reset_index(name="Inasistencias")
display(por_grupo)

sns.barplot(data=por_grupo, x="Grupo", y="Inasistencias", hue="Grupo", legend=False)
plt.title("Total de inasistencias por grupo")
plt.show()

## 2. Inasistencias por seguimiento

In [ ]:
por_seg = df.groupby("Seguimiento").size().reset_index(name="Inasistencias")
display(por_seg)

sns.barplot(data=por_seg, y="Seguimiento", x="Inasistencias", hue="Seguimiento", legend=False)
plt.title("Inasistencias por seguimiento")
plt.tight_layout()
plt.show()

## 3. Inasistencias por grupo + seguimiento

In [ ]:
por_grupo_seg = df.groupby(["Grupo", "Seguimiento"]).size().reset_index(name="Inasistencias")
display(por_grupo_seg)

g = sns.catplot(data=por_grupo_seg, x="Seguimiento", y="Inasistencias",
                col="Grupo", kind="bar", hue="Seguimiento", legend=False, sharey=False)
g.set_xticklabels(rotation=45)
plt.tight_layout()
plt.show()

## 4. Inasistencias por programa

In [ ]:
por_programa = df.groupby("Nombre_programa_limpio").size().reset_index(name="Inasistencias").sort_values("Inasistencias", ascending=False)
display(por_programa)

sns.barplot(data=por_programa, y="Nombre_programa_limpio", x="Inasistencias", hue="Nombre_programa_limpio", legend=False)
plt.title("Inasistencias por programa")
plt.tight_layout()
plt.show()

## 5. Inasistencias por sede

In [ ]:
por_sede = df.groupby("Sede").size().reset_index(name="Inasistencias").sort_values("Inasistencias", ascending=False)
display(por_sede)

sns.barplot(data=por_sede, x="Sede", y="Inasistencias", hue="Sede", legend=False)
plt.title("Inasistencias por sede")
plt.tight_layout()
plt.show()

## 6. Tabla cruzada: Programa × Seguimiento

In [ ]:
cruzada = df.pivot_table(index="Nombre_programa_limpio", columns="Seguimiento", aggfunc="size", fill_value=0)
display(cruzada)

plt.figure(figsize=(16, max(6, len(cruzada) * 0.4)))
sns.heatmap(cruzada, annot=True, fmt="d", cmap="Reds", linewidths=0.5)
plt.title("Inasistencias: Programa × Seguimiento")
plt.tight_layout()
plt.show()

## 7. Tabla cruzada: Sede × Seguimiento

In [ ]:
cruzada_sede = df.pivot_table(index="Sede", columns="Seguimiento", aggfunc="size", fill_value=0)
display(cruzada_sede)

plt.figure(figsize=(12, max(4, len(cruzada_sede) * 0.5)))
sns.heatmap(cruzada_sede, annot=True, fmt="d", cmap="Blues", linewidths=0.5)
plt.title("Inasistencias: Sede × Seguimiento")
plt.tight_layout()
plt.show()

## 8. Inasistencias por curso

In [ ]:
por_curso = df.groupby(["Nombre_curso", "Grupo"]).size().reset_index(name="Inasistencias").sort_values("Inasistencias", ascending=False)
display(por_curso.head(20))

top_cursos = por_curso.head(15)
sns.barplot(data=top_cursos, y="Nombre_curso", x="Inasistencias", hue="Grupo", dodge=False)
plt.title("Top 15 cursos con más inasistencias")
plt.tight_layout()
plt.show()

## 9. Resumen

In [ ]:
print("=== RESUMEN ===")
print(f"Total inasistencias: {len(df):,}")
print(f"Estudiantes: {df['Numero_identificacion_estudiante'].nunique():,}")
print(f"Justificadas: {df['Justificacion'].sum():,} ({df['Justificacion'].mean()*100:.1f}%)")
print(f"Cursos: {df['Nombre_curso'].nunique()}")
print(f"Programas: {df['Nombre_programa_limpio'].nunique()}")
print(f"Sedes: {df['Sede'].nunique()}")

In [ ]:
# Programas que quedaron después del filtro
print("Programas incluidos en el análisis:")
for p in sorted(df['Codigo_programa'].unique()):
    print(f"  - {p}")